## Setup

In [5]:
from dotenv import load_dotenv
from utils import chat_interface

In [6]:
load_dotenv()

True

## MCP tools and agents

In [7]:
# Mounts the CultPass and UDA-Hub MCP servers in memory (no processes, no network).
# The tools are async-only, so agents must be run with `await agent.ainvoke(...)`.
from agentic.mcp_servers.toolbox import build_default_toolbox
from agentic.agents.registry import build_agents

toolbox = build_default_toolbox()
await toolbox.connect()          # re-running this cell is safe; run the last cell to close it

In [8]:
# Each agent only gets the tools bound to it in agentic/agents/registry.py
agents = build_agents(toolbox)
for key, agent in agents.items():
    print(f"{key:20s} {agent.get_agent_tools()}")

classifier_agent     []
service_desk_agent   ['get_support_user', 'get_support_user_by_external_id', 'get_ticket_history', 'get_open_tickets', 'create_ticket', 'update_ticket', 'add_ticket_message']
resolver_agent       ['get_knowledge_articles']
account_ops_agent    ['lookup_user_account', 'lookup_user_by_name', 'lookup_user_by_email', 'get_reservations', 'get_subscriptions', 'get_experience', 'set_blocked', 'set_unblocked']
escalation_agent     ['escalate_to_human']


## Run

In [9]:
# Design notes: agentic/design/architecture.md (TODO: diagram and agent roles, Rubric 2)
# Agents:       agentic/agents/ (tools bound in agentic/agents/registry.py)
# MCP tools:    agentic/mcp_servers/
# Orchestrator: agentic/workflow.py

In [10]:
# The orchestrator needs the connected toolbox from the cells above, so it is built here.
# Conversation state is saved in solution/data/core/checkpoints.db (one thread per ticket
# conversation), so it survives a kernel restart.
from agentic.checkpointing import open_checkpointer, close_checkpointer
from agentic.workflow import build_orchestrator

checkpointer = await open_checkpointer()
orchestrator = await build_orchestrator(toolbox, checkpointer=checkpointer)

In [4]:
await chat_interface(orchestrator, "1")

User: Hi
Assistant: Hello! How can I assist you today?
User: q
Assistant: Goodbye!


In [5]:
# Everything saved for conversation "1": one snapshot per step, newest first
config = {"configurable": {"thread_id": "1"}}
history = [snapshot async for snapshot in orchestrator.aget_state_history(config)]
history[0].values["messages"]

[HumanMessage(content='Hi', additional_kwargs={}, response_metadata={}, id='43c56100-7a4e-4ff0-adb7-1fbbfdac82e3'),
 AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 60, 'total_tokens': 70, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_34a54ae93c', 'id': 'chatcmpl-C1loRWd5jRqktu5Fut6YZAZsTdc6S', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--0ab48c4a-6130-4dc9-96cd-271e2be7b7c8-0', usage_metadata={'input_tokens': 60, 'output_tokens': 10, 'total_tokens': 70, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})]

## Test scenarios

Six ticket types run end to end through the orchestrator. Each scenario is a scripted conversation, and the result is checked against the expected outcome (conversation phase, escalation, the ticket's status in the database, and the escalation reason).

In [11]:
import uuid

from langchain_core.messages import HumanMessage
from sqlalchemy import text

from agentic.mcp_servers.udahub.knowledge_index import create_udahub_engine
from agentic.mcp_servers.udahub.repository import UdaHubRepository

udahub_engine = create_udahub_engine()
repo = UdaHubRepository(udahub_engine)


def new_thread(name):
    # A new thread id per run: conversations are saved, so reusing an id would continue an old one.
    return f"{name}-{uuid.uuid4().hex[:6]}"


async def converse(thread_id, turns, show=True):
    """Send each message to the orchestrator on one thread, print the exchange, return the final state."""
    config = {"configurable": {"thread_id": thread_id}}
    for turn in turns:
        result = await orchestrator.ainvoke({"messages": [HumanMessage(turn)]}, config)
        if show:
            print(f"USER: {turn}\n\nBOT:  {result['messages'][-1].content}\n")
    return (await orchestrator.aget_state(config)).values


def ticket_status(ticket_id):
    with udahub_engine.connect() as conn:
        query = text("SELECT status FROM ticket_metadata WHERE ticket_id = :ticket_id")
        return conn.execute(query, {"ticket_id": ticket_id}).scalar()

In [12]:
# Expected outcome per scenario: how the conversation ends, whether a human was notified,
# the ticket status in udahub.db, and the reason code when escalated.
RESOLVED = {"phase": "closed", "escalated": False, "ticket_status": "resolved", "reason": None}


def escalated(reason):
    return {"phase": "escalated", "escalated": True, "ticket_status": "escalated", "reason": reason}


SCENARIOS = [
    {
        "name": "simple_faq",
        "type": "Simple FAQ question, answered from the knowledge base",
        "turns": [
            "How do I reserve a spot for an event? I was looking at the catalogue just now. "
            "I'm Cathy Bloom, cathy.bloom@florals.org. Please reply in Spanish and contact me by email.",
            "Yes, that answers it, thank you!",
        ],
        "expect": RESOLVED,
    },
    {
        "name": "technical_issue",
        "type": "Technical issue with a standard fix",
        "turns": [
            "My app crashes whenever I open it. It started this morning while I was opening my QR code. "
            "I'm Bob Stone, bob.stone@granite.com",
            "Yes, that fixed it, thanks!",
        ],
        "expect": RESOLVED,
    },
    {
        "name": "urgent_billing",
        "type": "Urgent, complex problem (double charge and locked out)",
        "turns": [
            "I was charged twice this month and now I can't log in at all, and my event is tomorrow! "
            "It happened a minute ago while I was paying. I'm Eva Green, eva.green@ecosoul.net",
        ],
        "expect": escalated("urgent_complex"),
    },
    {
        "name": "no_knowledge",
        "type": "Question the knowledge base cannot answer",
        "turns": [
            "Can I bring my pet iguana to the pottery workshop? I was browsing events just now. "
            "I'm Frank Ocean, frank.ocean@seawaves.io",
        ],
        "expect": escalated("low_confidence"),
    },
    {
        "name": "blocked_account",
        "type": "Customer with a blocked account",
        "turns": [
            "My app shows an error whenever I open events. It started yesterday while I was browsing. "
            "I'm Alice Kingsley, alice.kingsley@wonderland.com",
        ],
        "expect": escalated("account_blocked"),
    },
    {
        "name": "fix_failed",
        "type": "The suggested fix does not work",
        "turns": [
            "My app is very slow and freezes when I scroll. It started yesterday while I was browsing events. "
            "I'm Cathy Bloom, cathy.bloom@florals.org",
            "No, it is still just as slow.",
        ],
        "expect": escalated("fix_failed"),
    },
]

In [13]:
results = []
for scenario in SCENARIOS:
    print("=" * 100)
    print(f"{scenario['name']}: {scenario['type']}\n")
    thread_id = new_thread(scenario["name"])
    state = await converse(thread_id, scenario["turns"])
    actual = {
        "phase": state.get("phase"),
        "escalated": bool(state.get("escalated")),
        "ticket_status": ticket_status(state.get("ticket_id")),
        "reason": state.get("escalation_code"),
    }
    results.append(
        {
            "scenario": scenario["name"],
            "thread_id": thread_id,
            "passed": actual == scenario["expect"],
            "actual": actual,
            "expected": scenario["expect"],
        }
    )

simple_faq: Simple FAQ question, answered from the knowledge base

USER: How do I reserve a spot for an event? I was looking at the catalogue just now. I'm Cathy Bloom, cathy.bloom@florals.org. Please reply in Spanish and contact me by email.

BOT:  Hi Cathy! To reserve a spot for your event, please follow these steps:

1. Open the CultPass app.
2. Browse the experience catalog.
3. Tap on the event you want to reserve.
4. If it's a premium or limited event, check if you need reservation confirmation via email.
5. Remember to arrive at least 15 minutes early with your QR code visible.

Please try these steps and let me know if it solves your problem!

USER: Yes, that answers it, thank you!

BOT:  I'm glad that fixed it! Your ticket is now closed. Have a wonderful day!

technical_issue: Technical issue with a standard fix

USER: My app crashes whenever I open it. It started this morning while I was opening my QR code. I'm Bob Stone, bob.stone@granite.com

BOT:  Hi Bob,

I understand you'

In [14]:
print(f"{'scenario':18s} {'result':7s} outcome")
for r in results:
    a = r["actual"]
    outcome = f"{a['phase']}, ticket {a['ticket_status']}" + (f", escalated: {a['reason']}" if a["escalated"] else ", resolved from the knowledge base")
    print(f"{r['scenario']:18s} {'PASS' if r['passed'] else 'FAIL':7s} {outcome}")
    if not r["passed"]:
        print(f"{'':26s} expected {r['expected']}")

failed = [r["scenario"] for r in results if not r["passed"]]
assert not failed, f"scenarios that did not end as expected: {failed}"

scenario           result  outcome
simple_faq         PASS    closed, ticket resolved, resolved from the knowledge base
technical_issue    PASS    closed, ticket resolved, resolved from the knowledge base
urgent_billing     PASS    escalated, ticket escalated, escalated: urgent_complex
no_knowledge       PASS    escalated, ticket escalated, escalated: low_confidence
blocked_account    PASS    escalated, ticket escalated, escalated: account_blocked
fix_failed         PASS    escalated, ticket escalated, escalated: fix_failed


## Memory demo

Three kinds of memory are in use:
- **Short-term:** the conversation state kept per `thread_id` by the LangGraph checkpointer.
- **Persistent:** that state is stored in `data/core/checkpoints.db`, so it survives a restart.
- **Long-term:** what is learned about a customer is stored in the `customer_memory` table and used in later conversations.

In [15]:
# Short-term memory: one thread keeps its context from message to message.
short_term = new_thread("short_term")
state = await converse(short_term, ["Hi, my app keeps crashing."])
print("remembered so far:", state["intake"], "\n")

# The second message only adds the missing details. The issue from the first message is still known.
state = await converse(
    short_term,
    ["It started this morning while I was opening my QR code. I'm Bob Stone, bob.stone@granite.com"],
)
print("phase:", state["phase"], "| ticket:", state["ticket_id"])

USER: Hi, my app keeps crashing.

BOT:  I'm sorry you're having trouble, and I'd like to help. To get started, could you tell me:
1. When did it happen?
2. What were you doing when it happened?
3. What is your name and your email address?

remembered so far: {'issue': 'app keeps crashing'} 

USER: It started this morning while I was opening my QR code. I'm Bob Stone, bob.stone@granite.com

BOT:  Hi Bob,

I understand you're experiencing issues with the app crashing when opening your QR code. Here are some steps you can try:

1. Force close the app and then reopen it.
2. Restart your device.
3. Check for any pending updates in your app store and install them if available.
4. If the problem persists, clear the app cache.

Please try these steps and let me know if it solves the problem.

phase: awaiting_confirmation | ticket: b5f9fa39-dd4f-4848-ae31-ee91e6ca5de4


In [16]:
# Inspect any conversation by its thread id: the messages, the tool calls and the decisions.
from agentic.observability import read_events

config = {"configurable": {"thread_id": short_term}}
snapshot = await orchestrator.aget_state(config)
print("MESSAGES")
for message in snapshot.values["messages"]:
    print(f"  {message.type:5s} {str(message.content)[:90]!r}")

print("\nTOOL USAGE (from the event log)")
for event in read_events(thread_id=short_term):
    if event["event"] == "tool_call":
        print(f"  {event['agent']:24s} {event['tool']:34s} {event['outcome']}")

print("\nROUTING")
print("  " + " -> ".join(e["next"] for e in read_events(thread_id=short_term) if e["event"] == "routing_decision"))

MESSAGES
  human 'Hi, my app keeps crashing.'
  ai    "I'm sorry you're having trouble, and I'd like to help. To get started, could you tell me:\n"
  human "It started this morning while I was opening my QR code. I'm Bob Stone, bob.stone@granite.c"
  ai    "Hi Bob,\n\nI understand you're experiencing issues with the app crashing when opening your Q"

TOOL USAGE (from the event log)
  AccountOperationsAgent   lookup_user_by_email               ok
  ServiceDeskAgent         get_support_user_by_external_id    ok
  ServiceDeskAgent         create_ticket                      ok
  ServiceDeskAgent         update_ticket                      ok
  orchestrator             get_customer_memory                ok
  orchestrator             get_ticket_history                 ok
  ResolverAgent            get_knowledge_articles             ok
  ServiceDeskAgent         add_ticket_message                 ok

ROUTING
  intake -> deliver -> intake -> classify -> account_lookup -> create_ticket -> recall

In [17]:
# Persistent memory: close the checkpoint database and rebuild the orchestrator, as after a restart.
await close_checkpointer(checkpointer)
checkpointer = await open_checkpointer()
orchestrator = await build_orchestrator(toolbox, checkpointer=checkpointer)

config = {"configurable": {"thread_id": short_term}}
state = (await orchestrator.aget_state(config)).values
print("after reopening:", state["phase"], "|", len(state["messages"]), "messages | ticket", state["ticket_id"])

# The conversation carries on where it stopped: the customer confirms the fix.
state = await converse(short_term, ["Yes, it works now, thank you!"])
print("phase:", state["phase"], "| ticket status:", ticket_status(state["ticket_id"]))

after reopening: awaiting_confirmation | 4 messages | ticket b5f9fa39-dd4f-4848-ae31-ee91e6ca5de4
USER: Yes, it works now, thank you!

BOT:  I'm glad that fixed it! Your ticket is now closed. Have a wonderful day!

phase: closed | ticket status: resolved


In [18]:
# Long-term memory: what was learned is stored per customer, across conversations.
def show_memory(external_user_id):
    support_user = repo.get_user_by_external_id(external_user_id)
    for item in repo.get_customer_memories(support_user["user_id"]):
        print(f"  {item['memory_type']:15s} {item['key']:18s} {item['value']}")


print("Cathy (from the simple_faq scenario):")
show_memory("88382b")
print("\nBob (from the conversation above):")
show_memory("f556c0")

Cathy (from the simple_faq scenario):
  preference      preferred_contact  email
  preference      language           Spanish
  resolved_issue  reservation        reserve a spot for an event (fixed with: How to Reserve a Spot for an Event, Understanding CultPass Premium Experiences, Refund Policy Overview)

Bob (from the conversation above):
  preference      preferred_contact  email
  resolved_issue  technical          app keeps crashing (fixed with: CultPass App Not Opening, Troubleshooting App Performance)
  resolved_issue  technical          app crashes (fixed with: CultPass App Not Opening, Troubleshooting App Performance)


In [19]:
# A returning customer. Cathy asked for replies in Spanish in an earlier conversation,
# and the new conversation is on a new thread, so only long-term memory can know that.
state = await converse(
    new_thread("returning_customer"),
    [
        "The app freezes when I scroll through the events list. It started yesterday while I was browsing. "
        "I'm Cathy Bloom, cathy.bloom@florals.org"
    ],
)
print("recalled:", state["memory"]["preferences"], "|", len(state["memory"]["resolved_issues"]), "earlier resolved issue(s)")

USER: The app freezes when I scroll through the events list. It started yesterday while I was browsing. I'm Cathy Bloom, cathy.bloom@florals.org

BOT:  Hola Cathy,

He encontrado información útil sobre el problema de la aplicación que se congela al desplazarte por la lista de eventos. Aquí tienes algunos pasos que puedes seguir:

1. Verifica tu conexión a internet para asegurarte de que sea estable.
2. Borra la caché y los datos de la aplicación.
3. Asegúrate de estar utilizando la última versión de la aplicación. Revisa si hay actualizaciones en tu tienda de aplicaciones.
4. Si el problema persiste, considera desinstalar y reinstalar la aplicación.

Por favor, intenta estos pasos y dime si resuelven el problema.

recalled: {'preferred_contact': 'email', 'language': 'Spanish'} | 1 earlier resolved issue(s)


In [20]:
# Memory in a decision: a customer whose problem was resolved twice before, and who comes back
# with the same kind of problem, goes to a human. Bob has one resolved technical issue from above.
# Add a second earlier one, then report a new technical problem.
bob = repo.get_user_by_external_id("f556c0")
repo.save_customer_memory(
    bob["user_id"], "resolved_issue", "technical", "App freezing (fixed with: Troubleshooting App Performance)"
)

state = await converse(
    new_thread("recurring_issue"),
    [
        "My app crashes again as soon as I open it. It started today while I was opening my tickets. "
        "I'm Bob Stone, bob.stone@granite.com"
    ],
)
print("escalated:", state["escalated"], "| reason:", state["escalation_code"], "| knowledge base searched:", state.get("kb") is not None)

USER: My app crashes again as soon as I open it. It started today while I was opening my tickets. I'm Bob Stone, bob.stone@granite.com

BOT:  Hi Bob,

Thank you for reaching out to us again. I’m sorry to hear that you’re experiencing issues with the app. I’ve notified a member of our team, and they will follow up with you shortly regarding your ticket reference: d5acb4a6-fd23-4254-925a-dd5d0871b2ee.

We appreciate your patience and understanding.

Best regards,  
CultPass Support Team

escalated: True | reason: repeat_issue | knowledge base searched: False


## Logs and metrics

Every routing decision, classification, knowledge base search, tool call, escalation and outcome is written as one JSON line to `logs/agent_events.jsonl`.

In [21]:
import json

from agentic.observability import LOG_PATH, summarize_logs

print(LOG_PATH.relative_to(LOG_PATH.parents[1]), "\n")
for line in LOG_PATH.read_text().splitlines()[:3]:
    print(line)

logs/agent_events.jsonl 

{"ts": "2026-11-23T03:19:44.537+00:00", "level": "info", "event": "routing_decision", "thread_id": "simple_faq-9d3731", "node": "orchestrator", "phase": "intake", "next": "intake", "escalated": false}
{"ts": "2026-11-23T03:19:46.061+00:00", "level": "info", "event": "intake_checked", "thread_id": "simple_faq-9d3731", "node": "intake", "missing": [], "complete": true, "first_turn": true}
{"ts": "2026-11-23T03:19:46.061+00:00", "level": "info", "event": "node_completed", "thread_id": "simple_faq-9d3731", "node": "intake", "duration_ms": 1519}


In [22]:
metrics = summarize_logs()

kb = metrics["knowledge_base"]
out = metrics["outcomes"]
print(f"Knowledge base: {kb['searches']} searches, {kb['confident']} confident ({kb['success_rate']:.0%} success)")
print(f"Outcomes: {out['resolved']} resolved, {out['escalated']} escalated ({out['resolved_share']:.0%} resolved)")
print("Escalation reasons:", out["escalation_reasons"])
print("Ticket categories:", metrics["classification"])

print("\nTool usage (calls / not found / errors):")
for tool, counts in metrics["tool_usage"].items():
    print(f"  {tool:34s} {counts.get('calls', 0):3d} {counts.get('not_found', 0):3d} {counts.get('errors', 0):3d}")

print("\nTool usage by agent:")
for agent, tools in metrics["tool_usage_by_agent"].items():
    print(f"  {agent}: {tools}")

print("\nRouting decisions:", metrics["routing"])
print("Average time per step (ms):", metrics["avg_node_ms"])

Knowledge base: 7 searches, 6 confident (86% success)
Outcomes: 3 resolved, 5 escalated (38% resolved)
Escalation reasons: {'urgent_complex': 1, 'low_confidence': 1, 'account_blocked': 1, 'fix_failed': 1, 'repeat_issue': 1}
Ticket categories: {'reservation': 1, 'technical': 6, 'billing': 1, 'other': 1}

Tool usage (calls / not found / errors):
  add_ticket_message                  17   0   0
  create_ticket                        9   0   0
  escalate_to_human                    5   0   0
  get_customer_memory                  9   0   0
  get_knowledge_articles               7   0   0
  get_support_user_by_external_id      9   4   0
  get_ticket_history                   9   0   0
  lookup_user_by_email                 9   0   0
  save_customer_memory                 7   0   0
  update_ticket                       17   0   0

Tool usage by agent:
  AccountOperationsAgent: {'lookup_user_by_email': 9}
  EscalationAgent: {'escalate_to_human': 5}
  ResolverAgent: {'get_knowledge_articles': 

In [23]:
# The log must not expose customer data: no emails, names or message text.
log_text = LOG_PATH.read_text()
customers = [json.loads(line) for line in open("data/external/cultpass_users.jsonl")]
leaks = [value for c in customers for value in (c["email"], c["name"]) if value in log_text]
assert not leaks, f"customer data found in the log: {leaks}"
print(f"Checked {len(log_text.splitlines())} log lines for {len(customers)} customers' names and emails: none found.")

Checked 423 log lines for 6 customers' names and emails: none found.


## Cleanup

In [24]:
await close_checkpointer(checkpointer)
await toolbox.aclose()